In [1]:

#  Install Dependencies & Setup Device

!pip install -q speechbrain torchaudio

import os
import json
import random
from pathlib import Path
import torch
import torchaudio

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using PyTorch {torch.__version__} on device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 27.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 38.6 MB/s eta 0:00:00
Using PyTorch 2.10.0+cu128 on device: cuda


In [2]:

#  Scan Dataset & Create JSON Manifests

INPUT_DIR = Path("/kaggle/input/")
OUTPUT_DIR = Path("/kaggle/working/")

# Scan all subdirectories for audio files
audio_extensions = {".flac", ".wav", ".mp3"}
audio_files = []

for root, _, files in os.walk(INPUT_DIR):
    for f in files:
        if Path(f).suffix.lower() in audio_extensions:
            audio_files.append(Path(root) / f)

print(f"Total audio files found: {len(audio_files)}")

if len(audio_files) == 0:
    raise RuntimeError("No audio files found! Verify the LibriSpeech dataset has finished downloading in the sidebar.")

# Build utterance dictionary mapping speaker IDs
manifest = {}
for idx, filepath in enumerate(audio_files):
    # LibriSpeech hierarchy: .../speaker_id/chapter_id/speaker_chapter_utt.flac
    speaker_id = filepath.parent.parent.name if len(filepath.parents) > 2 else filepath.parent.name
    
    manifest[f"utt_{idx}"] = {
        "wav": str(filepath.resolve()),
        "speaker_id": speaker_id
    }

# Shuffle and perform an 85/15 train/valid split
items = list(manifest.items())
random.seed(42)
random.shuffle(items)

split_idx = int(len(items) * 0.85)
train_data = dict(items[:split_idx])
valid_data = dict(items[split_idx:])

with open(OUTPUT_DIR / "train.json", "w") as f:
    json.dump(train_data, f, indent=2)
with open(OUTPUT_DIR / "valid.json", "w") as f:
    json.dump(valid_data, f, indent=2)

print(f"\nManifests created successfully:")
print(f" - Train set: {len(train_data)} utterances")
print(f" - Validation set: {len(valid_data)} utterances")

Total audio files found: 113098

Manifests created successfully:
 - Train set: 96133 utterances
 - Validation set: 16965 utterances


In [3]:

#  PyTorch Dataset Class for Speaker Pair Comparison
from speechbrain.inference.speaker import EncoderClassifier
from torch.utils.data import Dataset, DataLoader

class VoxCelebPairDataset(Dataset):
    def __init__(self, manifest_path, sample_rate=16000, target_duration=3.0):
        with open(manifest_path, "r") as f:
            self.data = list(json.load(f).values())
            
        self.sample_rate = sample_rate
        self.max_samples = int(sample_rate * target_duration)
        
        self.speaker_to_files = {}
        for entry in self.data:
            spk = entry["speaker_id"]
            if spk not in self.speaker_to_files:
                self.speaker_to_files[spk] = []
            self.speaker_to_files[spk].append(entry["wav"])
            
        self.speakers = list(self.speaker_to_files.keys())

    def _load_and_pad(self, path):
        signal, sr = torchaudio.load(path)
        if sr != self.sample_rate:
            signal = torchaudio.transforms.Resample(sr, self.sample_rate)(signal)
            
        signal = signal.mean(dim=0)  # Convert to mono
        
        if signal.shape[0] < self.max_samples:
            padding = self.max_samples - signal.shape[0]
            signal = torch.nn.functional.pad(signal, (0, padding))
        else:
            signal = signal[:self.max_samples]
            
        return signal

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        anchor_entry = self.data[idx]
        anchor_spk = anchor_entry["speaker_id"]
        anchor_wav = self._load_and_pad(anchor_entry["wav"])
        
        is_same = random.choice([True, False])
        
        if is_same and len(self.speaker_to_files[anchor_spk]) > 1:
            pair_wav_path = random.choice([p for p in self.speaker_to_files[anchor_spk] if p != anchor_entry["wav"]])
            label = 1.0  # Same speaker label
        else:
            diff_spk = random.choice([s for s in self.speakers if s != anchor_spk])
            pair_wav_path = random.choice(self.speaker_to_files[diff_spk])
            label = -1.0  # Different speaker label
            
        pair_wav = self._load_and_pad(pair_wav_path)
        return anchor_wav, pair_wav, torch.tensor(label, dtype=torch.float32)

In [4]:

#  Load Pretrained ECAPA-TDNN & DataLoaders


# Load ECAPA-TDNN speaker model
classifier = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="/tmp/ecapa",
    run_opts={"device": device}
)

model = classifier.mods.embedding_model.to(device)

# Instantiate DataLoaders
train_dataset = VoxCelebPairDataset("/kaggle/working/train.json")
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

print(f"DataLoader ready with {len(train_loader)} batches per epoch.")

hyperparams.yaml: 0.00B [00:00, ?B/s]

embedding_model.ckpt:   0%|          | 0.00/83.3M [00:00<?, ?B/s]

mean_var_norm_emb.ckpt:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

classifier.ckpt:   0%|          | 0.00/5.53M [00:00<?, ?B/s]

label_encoder.txt: 0.00B [00:00, ?B/s]

Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


DataLoader ready with 6009 batches per epoch.


In [20]:

#  Fine-Tuning Execution (Pure PyTorch Engine)

import torch
import torch.nn as nn
import torch.optim as optim

# Extract underlying PyTorch model modules
embedding_model = classifier.mods.embedding_model
compute_features = classifier.mods.compute_features

# Unfreeze parameters and set model to training mode
for param in embedding_model.parameters():
    param.requires_grad = True

embedding_model.train()
compute_features.eval()  # Keep feature extractor frozen in eval mode

optimizer = optim.Adam(embedding_model.parameters(), lr=1e-5)
criterion = nn.CosineEmbeddingLoss(margin=0.2)

NUM_EPOCHS = 3
print(f"Starting Fine-Tuning for {NUM_EPOCHS} Epochs...\n")

for epoch in range(NUM_EPOCHS):
    total_loss = 0.0
    for batch_idx, (audio1, audio2, label) in enumerate(train_loader):
        # Format input shapes to 2D [batch_size, time_samples]
        if audio1.ndim > 2:
            audio1 = audio1.squeeze(1)
        if audio2.ndim > 2:
            audio2 = audio2.squeeze(1)

        # Unpack list and send individual tensors to device
        audio1, audio2, label = audio1.to(device), audio2.to(device), label.to(device)

        optimizer.zero_grad()

        # 1. Extract features (no gradients needed for mel-spectrogram computation)
        with torch.no_grad():
            feats1 = compute_features(audio1)
            feats2 = compute_features(audio2)

        # 2. Forward pass through ECAPA backbone
        emb1 = embedding_model(feats1).squeeze()
        emb2 = embedding_model(feats2).squeeze()

        # 3. Compute Cosine Loss & Backpropagate
        loss = criterion(emb1, emb2, label)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 50 == 0:
            print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}] | Step [{batch_idx}/{len(train_loader)}] | Loss: {loss.item():.4f}")

    print(f"\n--- Epoch {epoch + 1} Finished | Avg Loss: {total_loss / len(train_loader):.4f} ---\n")

# Save Fine-Tuned Model Weights
torch.save(embedding_model.state_dict(), "/kaggle/working/ecapa_librispeech_finetuned.pt")
print("Training Complete! Fine-tuned model saved to /kaggle/working/ecapa_librispeech_finetuned.pt")

Starting Fine-Tuning for 3 Epochs...

Epoch [1/3] | Step [0/6009] | Loss: 0.1964
Epoch [1/3] | Step [50/6009] | Loss: 0.1859
Epoch [1/3] | Step [100/6009] | Loss: 0.2119
Epoch [1/3] | Step [150/6009] | Loss: 0.2149
Epoch [1/3] | Step [200/6009] | Loss: 0.2649
Epoch [1/3] | Step [250/6009] | Loss: 0.3610
Epoch [1/3] | Step [300/6009] | Loss: 0.2455
Epoch [1/3] | Step [350/6009] | Loss: 0.2448
Epoch [1/3] | Step [400/6009] | Loss: 0.1843
Epoch [1/3] | Step [450/6009] | Loss: 0.3263
Epoch [1/3] | Step [500/6009] | Loss: 0.2193
Epoch [1/3] | Step [550/6009] | Loss: 0.2652
Epoch [1/3] | Step [600/6009] | Loss: 0.2092
Epoch [1/3] | Step [650/6009] | Loss: 0.1922
Epoch [1/3] | Step [700/6009] | Loss: 0.2964
Epoch [1/3] | Step [750/6009] | Loss: 0.2861
Epoch [1/3] | Step [800/6009] | Loss: 0.1922
Epoch [1/3] | Step [850/6009] | Loss: 0.2292
Epoch [1/3] | Step [900/6009] | Loss: 0.1934
Epoch [1/3] | Step [950/6009] | Loss: 0.2325
Epoch [1/3] | Step [1000/6009] | Loss: 0.1933
Epoch [1/3] | Step 

In [21]:

#  Model Evaluation & Similarity Scoring

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

# Instantiate validation dataset
valid_dataset = VoxCelebPairDataset("/kaggle/working/valid.json")
valid_loader = DataLoader(valid_dataset, batch_size=16, shuffle=False)

# Load fine-tuned weights into the embedding model
embedding_model = classifier.mods.embedding_model
embedding_model.load_state_dict(torch.load("/kaggle/working/ecapa_librispeech_finetuned.pt"))
embedding_model.eval()
compute_features = classifier.mods.compute_features
compute_features.eval()

total_sim = 0.0
correct_preds = 0
total_samples = 0

print("Evaluating Fine-Tuned Model on Validation Set...\n")

with torch.no_grad():
    for audio1, audio2, label in valid_loader:
        if audio1.ndim > 2:
            audio1 = audio1.squeeze(1)
        if audio2.ndim > 2:
            audio2 = audio2.squeeze(1)

        audio1, audio2, label = audio1.to(device), audio2.to(device), label.to(device)

        # Extract features & embeddings
        feats1 = compute_features(audio1)
        feats2 = compute_features(audio2)
        
        emb1 = embedding_model(feats1).squeeze()
        emb2 = embedding_model(feats2).squeeze()

        # Calculate Cosine Similarity (-1.0 to +1.0)
        cos_sim = F.cosine_similarity(emb1, emb2)

        # Threshold at 0.25 similarity score for match classification
        predictions = torch.where(cos_sim > 0.25, 1.0, -1.0)
        correct_preds += (predictions == label).sum().item()
        total_samples += label.size(0)

accuracy = (correct_preds / total_samples) * 100
print(f"Validation Accuracy: {accuracy:.2f}% ({correct_preds}/{total_samples} pairs correctly classified)")

Evaluating Fine-Tuned Model on Validation Set...

Validation Accuracy: 87.42% (14830/16965 pairs correctly classified)


In [22]:

#  Custom Voice Verification Inference Function

import torch
import torchaudio
import torch.nn.functional as F

def verify_speakers(file_path1, file_path2, threshold=0.25):
    """
    Compares two audio files and determines if they belong to the same speaker.
    """
    embedding_model.eval()
    compute_features.eval()

    def process_audio(path, sample_rate=16000, max_duration=3.0):
        signal, sr = torchaudio.load(path)
        if sr != sample_rate:
            signal = torchaudio.transforms.Resample(sr, sample_rate)(signal)
        signal = signal.mean(dim=0)  # Mono
        max_samples = int(sample_rate * max_duration)
        if signal.shape[0] < max_samples:
            padding = max_samples - signal.shape[0]
            signal = F.pad(signal, (0, padding))
        else:
            signal = signal[:max_samples]
        return signal.unsqueeze(0).to(device)

    with torch.no_grad():
        # Load and pad audio
        wav1 = process_audio(file_path1)
        wav2 = process_audio(file_path2)

        # Extract features and embeddings
        feats1 = compute_features(wav1)
        feats2 = compute_features(wav2)
        emb1 = embedding_model(feats1).squeeze()
        emb2 = embedding_model(feats2).squeeze()

        # Compute cosine similarity
        sim_score = F.cosine_similarity(emb1.unsqueeze(0), emb2.unsqueeze(0)).item()

    is_same = sim_score > threshold
    verdict = "MATCH (Same Speaker)" if is_same else "MISMATCH (Different Speakers)"
    
    print(f"File 1: {file_path1}")
    print(f"File 2: {file_path2}")
    print(f"Similarity Score: {sim_score:.4f}")
    print(f"Verification Result: {verdict}\n")
    return is_same, sim_score

#  Test on Two Samples from Validation Manifest 
import json
with open("/kaggle/working/valid.json", "r") as f:
    valid_samples = list(json.load(f).values())

sample_audio_1 = valid_samples[0]["wav"]
sample_audio_2 = valid_samples[1]["wav"]

print("--- Testing Inference Pipeline ---")
verify_speakers(sample_audio_1, sample_audio_2)

--- Testing Inference Pipeline ---
File 1: /kaggle/input/datasets/pypiahmad/librispeech-asr-corpus/train-clean-360/LibriSpeech/train-clean-360/38/121024/38-121024-0112.flac
File 2: /kaggle/input/datasets/pypiahmad/librispeech-asr-corpus/train-clean-360/LibriSpeech/train-clean-360/258/130878/258-130878-0050.flac
Similarity Score: -0.1712
Verification Result: MISMATCH (Different Speakers)



(False, -0.17120878398418427)